# Chapter 2 Practical 05: SBERT Semantic Recommender

Learning objectives:
- Explain lexical similarity versus semantic similarity.
- Encode movie descriptions with SBERT when available.
- Fall back to TF-IDF when `sentence-transformers` or the model is unavailable.
- Compare TF-IDF and semantic recommendation results.

Slide connection: deep content models, SBERT/BERT embeddings, semantic similarity, and practical fallback design.


TF-IDF works with shared words. SBERT can also capture related meanings, such as `astronaut`, `space`, `orbit`, and `Mars`.


In [1]:
import pandas as pd
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_02_content_based/data"),
]

for data_dir in DATA_DIRS:
    csv_path = data_dir / "movies_chapter2.csv"
    if csv_path.exists():
        movies = pd.read_csv(csv_path)
        break
else:
    url = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_02_content_based/data/movies_chapter2.csv"
    movies = pd.read_csv(url)

movies.head()

,movie_id,title,genres,director,year,duration_min,rating,family_friendly,description,keywords
0,1,Inception,Sci-Fi|Thriller|Action,Christopher Nolan,2010,148,8.8,0,A thief enters layered dreams to plant an idea...,dreams heist subconscious mind-bending
1,2,Interstellar,Sci-Fi|Adventure|Drama,Christopher Nolan,2014,169,8.7,0,Astronauts travel through a wormhole to find a...,space exploration wormhole survival family
2,3,Titanic,Romance|Drama,James Cameron,1997,195,7.9,0,A young couple from different social classes f...,romance ship tragedy historical
3,4,The Matrix,Sci-Fi|Action,The Wachowskis,1999,136,8.7,0,A hacker discovers that reality is a simulated...,simulation hacker reality action cyberpunk
4,5,Toy Story,Animation|Adventure|Comedy|Family,John Lasseter,1995,81,8.3,1,A cowboy doll feels threatened when a space ra...,toys friendship family adventure


First build a TF-IDF baseline that always works in a basic Python environment.


In [2]:
# Teaching note: Try SBERT embeddings first; fall back to TF-IDF if the optional package is unavailable.
# TF-IDF converts text into weighted numeric features.
from sklearn.feature_extraction.text import TfidfVectorizer
# Cosine similarity turns vectors into pairwise recommendation scores.
from sklearn.metrics.pairwise import cosine_similarity

movies["semantic_text"] = movies["title"] + ". " + movies["description"] + " Keywords: " + movies["keywords"]

# TF-IDF converts text into weighted numeric features.
tfidf = TfidfVectorizer(stop_words="english") #creates the TF-IDF converter and tells it to remove common English words.

tfidf_matrix = tfidf.fit_transform(movies["semantic_text"]) #applies the converter to movie texts and creates the numeric TF-IDF matrix.
# Cosine similarity turns vectors into pairwise recommendation scores.
tfidf_similarity = cosine_similarity(tfidf_matrix)


In [3]:
# Show TF-IDF table: rows = movies, columns = terms, values = TF-IDF weights
tfidf_table = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=movies["title"],
    columns=tfidf.get_feature_names_out()
)

tfidf_table.round(2)

,action,actor,adventure,ambition,artistic,aspiring,astronaut,astronauts,barriers,batman,...,threatened,titanic,toy,toys,tragedy,travel,uses,world,wormhole,young
title,,,,,,,,,,,,,,,,,,,,,
Inception,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Interstellar,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.25,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.29,0.00,0.00,0.58,0.00
Titanic,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.28,0.00,0.00,0.28,0.00,0.00,0.00,0.00,0.24
The Matrix,0.24,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.24,0.00,0.00
Toy Story,0.00,0.00,0.22,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.26,0.00,0.51,0.26,0.00,0.00,0.00,0.00,0.00,0.00
Finding Nemo,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
The Dark Knight,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.28,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
The Martian,0.00,0.00,0.00,0.00,0.00,0.00,0.41,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.24,0.00,0.00,0.00
The Notebook,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.28,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Now try SBERT. If the package is missing or the model cannot be downloaded, the notebook continues with the TF-IDF fallback.


In [4]:
# Teaching note: Compute semantic similarity between movies from the chosen embedding matrix.
embedding_source = "tfidf fallback" #Sets the default method name to TF-IDF fallback.
sbert_similarity = tfidf_similarity #Uses TF-IDF similarity as the default similarity matrix.

try:
    from sentence_transformers import SentenceTransformer #Imports the SBERT library.
    model = SentenceTransformer("all-MiniLM-L6-v2") #Loads a small pre-trained sentence embedding model.
    embeddings = model.encode(movies["semantic_text"].tolist(), show_progress_bar=True) #Converts each movie text into a dense semantic embedding vector.
    # Cosine similarity turns vectors into pairwise recommendation scores.
    sbert_similarity = cosine_similarity(embeddings)
    embedding_source = "sentence-transformers/all-MiniLM-L6-v2"
except Exception as exc:
    print("SBERT is not available in this environment.")
    print("Using TF-IDF fallback instead.")
    print(type(exc).__name__, str(exc)[:160])

embedding_source

/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

'sentence-transformers/all-MiniLM-L6-v2'

In [5]:
#Each movie is now represented as a dense numerical vector. These numbers do not have direct word meanings, but together they capture semantic meaning.

embedding_table = pd.DataFrame(
    embeddings,
    index=movies["title"]
)

embedding_table.iloc[:, :8].round(3)

,0,1,2,3,4,5,6,7
title,,,,,,,,
Inception,-0.040,0.004,-0.001,0.027,0.024,0.025,0.053,0.016
Interstellar,-0.053,-0.096,0.011,0.057,-0.038,-0.098,0.060,-0.042
Titanic,-0.054,0.021,0.031,0.108,-0.001,0.068,-0.011,0.045
The Matrix,-0.018,-0.026,-0.145,-0.044,-0.053,-0.023,-0.014,-0.046
Toy Story,-0.041,-0.037,0.046,0.044,-0.009,0.015,0.040,0.036
Finding Nemo,-0.058,0.043,-0.021,-0.032,0.020,-0.015,0.069,0.024
The Dark Knight,-0.008,0.009,-0.151,-0.013,-0.005,0.062,-0.023,0.008
The Martian,0.031,-0.013,-0.044,0.065,0.088,-0.050,0.083,0.036
The Notebook,0.012,-0.009,-0.019,0.074,-0.042,0.087,-0.000,-0.006


Use the same recommendation function for either similarity matrix.


In [6]:
# Teaching note: Rank semantically similar movies for a selected title.
#title = the movie we start from, for example "Interstellar"

#similarity_matrix = movie–movie similarity table, for example sbert_similarity

#n=5 = return 5 recommendations by default
def recommend(title, similarity_matrix, n=5):
    idx = movies.index[movies["title"].eq(title)][0]
    scores = sorted(enumerate(similarity_matrix[idx]), key=lambda x: x[1], reverse=True)
    #Gets similarity scores between Interstellar and all other movies.
    return pd.DataFrame([
        {"query_movie": title, "recommended_movie": movies.loc[i, "title"], "score": round(float(score), 3)}
        for i, score in scores[1:n+1]
    ])

recommend("Interstellar", sbert_similarity)


,query_movie,recommended_movie,score
0,Interstellar,Gravity,0.557
1,Interstellar,The Martian,0.522
2,Interstellar,Titanic,0.394
3,Interstellar,Finding Nemo,0.329
4,Interstellar,Toy Story,0.300


Compare lexical TF-IDF results with semantic SBERT results. If SBERT is not available, both columns will show the fallback behavior.


In [7]:
# Teaching note: Search with a natural-language query by embedding the query in the same space.
tfidf_results = recommend("Interstellar", tfidf_similarity, n=5).rename(columns={
    "recommended_movie": "tfidf_recommendation",
    "score": "tfidf_score",
})
sbert_results = recommend("Interstellar", sbert_similarity, n=5).rename(columns={
    "recommended_movie": "semantic_recommendation",
    "score": "semantic_score",
})

pd.concat([
    tfidf_results[["tfidf_recommendation", "tfidf_score"]],
    sbert_results[["semantic_recommendation", "semantic_score"]],
], axis=1)


,tfidf_recommendation,tfidf_score,semantic_recommendation,semantic_score
0,Gravity,0.171,Gravity,0.557
1,Toy Story,0.133,The Martian,0.522
2,Paddington,0.122,Titanic,0.394
3,The Martian,0.080,Finding Nemo,0.329
4,Finding Nemo,0.043,Toy Story,0.300


Zero-shot style semantic search uses a text query instead of an input movie.


In [8]:
# Teaching note: Compare semantic search behavior with keyword-style TF-IDF behavior.
queries = ["movies about space exploration", "romantic drama about lifelong love"]

if embedding_source.startswith("sentence-transformers"):
    query_embeddings = model.encode(queries, show_progress_bar=False) #query_embeddings = model.encode(queries, show_progress_bar=False)
    # Cosine similarity turns vectors into pairwise recommendation scores.
    query_scores = cosine_similarity(query_embeddings, embeddings)
else:
    query_matrix = tfidf.transform(queries)
    # Cosine similarity turns vectors into pairwise recommendation scores.
    query_scores = cosine_similarity(query_matrix, tfidf_matrix)

rows = []
for q_idx, query in enumerate(queries):
    best = query_scores[q_idx].argsort()[::-1][:4]
    for movie_idx in best:
        rows.append({"query": query, "movie": movies.loc[movie_idx, "title"], "score": round(float(query_scores[q_idx, movie_idx]), 3)})

pd.DataFrame(rows)

,query,movie,score
0,movies about space exploration,Interstellar,0.538
1,movies about space exploration,Titanic,0.388
2,movies about space exploration,The Martian,0.375
3,movies about space exploration,Gravity,0.371
4,romantic drama about lifelong love,The Notebook,0.640
5,romantic drama about lifelong love,Titanic,0.577
6,romantic drama about lifelong love,La La Land,0.351
7,romantic drama about lifelong love,Toy Story,0.296


## What did we learn?

- TF-IDF is lexical: shared words drive similarity.
- SBERT is semantic: related meanings can be close even with different words.
- Optional models should have fallback logic so teaching notebooks still run.

## Challenge Lab

1. Try the query mind bending simulated reality in the zero-shot search section. Compare the result with a more literal keyword query and explain which movies move up or down.
2. Add a new movie description that uses synonyms instead of exact genre words, then rerun the recommender. Does the semantic method or TF-IDF method handle it better?
